In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.io import loadmat  # Pour charger les fichiers .mat
import sys
import PySpin
import flir_tools
import flir_video
import flir_image
import SLMcontrol
import Fonctions
import time
from scipy.ndimage import zoom
import threading
from PyQt5.QtWidgets import QApplication, QMainWindow, QLabel, QDesktopWidget, QVBoxLayout, QWidget
from PyQt5.QtGui import QPixmap, QImage
from PyQt5.QtCore import Qt, QTimer, QObject, pyqtSignal
%matplotlib qt

### Camera FLIR

In [2]:
#Camera initialization

sys, clist, cam = flir_tools.connect_cam()
DXmax = 1280;     # Maximum width of the frame (pixels)
DYmax = 1024;     # Maximum height of the frame (pixels)
Xinmax = 0;       # Horizontal offset (starting X-coordinate)
Yinmax = 0;       # Vertical offset (starting Y-coordinate)
tempsexp = 7;

In [10]:
# Video

tempsexp = 7
flir_video.live_view(cam, tempsexp)

>>> LIVE en cours... Fermez la fenêtre pour ARRETER le processus.

[!] Fenêtre fermée. Interruption volontaire du script.


SystemExit: Script stoppé par l'utilisateur.

In [7]:
# Image

image = flir_image.capture(cam, tempsexp) 

plt.figure()  
plt.imshow(image, cmap='turbo')
plt.colorbar()
plt.title('Image')
plt.show()

In [3]:
# Camera ROI

data = flir_image.capture(cam, tempsexp)

DXff, DYff = 624, 624
Xinff, Yinff = 0, 0

if data is not None:
    h = plt.figure()
    plt.imshow(data, cmap='turbo')
    plt.title("Click on ROI center")
    pts = plt.ginput(1) 
    plt.close(h)
    
    if pts:
        xcenter, ycenter = pts[0]
        Xinff = int(np.floor(xcenter) - np.floor(DXff / 2))
        Yinff = int(np.floor(ycenter) - np.floor(DYff / 2))
        Xinff = max(0, min(Xinff, DXmax - DXff))
        Yinff = max(0, min(Yinff, DYmax - DYff))
        Xinff = 8 * (Xinff // 8)
        Yinff = 8 * (Yinff // 8)
        tempsexp = flir_video.live_view(cam, tempsexp, DXff, DYff, Xinff, Yinff)

>>> LIVE en cours... Fermez la fenêtre pour ARRETER le processus.

[!] Fenêtre fermée. Interruption volontaire du script.


SystemExit: Script stoppé par l'utilisateur.

c:\Users\Manip_1\anaconda3\envs\flir_env\lib\site-packages\IPython\core\interactiveshell.py:3585: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [7]:
image = flir_image.capture(cam, tempsexp, DXff, DYff, Xinff, Yinff) 

plt.figure()  
plt.imshow(image, cmap='turbo')
plt.colorbar()
plt.title('Image')
plt.show()

### SLM Holoeye

In [4]:
# SLM Initilization

controller = SLMcontrol.init_slm_display()
slm_width = controller.slm_window.width()
slm_height = controller.slm_window.height()

A0 = np.zeros((slm_height, slm_width), dtype=np.uint8)

In [5]:
# SLM ROI

Cxslm = 300
Dxslm = 500
Cyslm = 400
Dyslm = 500

Nrand = 50
SLMrand = np.floor(np.random.rand(Nrand, Nrand) + 0.5).astype(np.uint8)

SLMrand = Fonctions.calculCentreContour(Cxslm, Dxslm, Cyslm, Dyslm, SLMrand, A0)
SLMrand = (SLMrand.astype(np.uint16) * 100) 
SLMrand = np.clip(SLMrand, 0, 255).astype(np.uint8) 

SLMcontrol.update_slm_image(SLMrand)
time.sleep(0.2)

# 1. SLM calibration

1. Take reference speckle with blank SLM

In [6]:
SLMcontrol.update_slm_image(A0)
reference_data = flir_image.capture(cam, tempsexp, DXff, DYff, Xinff, Yinff)

2. Define the SLM random phase mask

In [7]:
h,w = 20, 20
random_matrix = np.random.rand(h,w)
M0 = (random_matrix > 0.5).astype(np.uint8)
M_display = A0.copy() # don't modify A0
M = Fonctions.calculCentreContour(960, 1900, 540, 1060, M0, M_display)

3. SLM calibration loop

In [8]:
vec, C = np.arange(0,256, 5), []

for k in range(len(vec)):
    M1 = M*vec[k]
    SLMcontrol.update_slm_image(M1)
    QApplication.processEvents()
    time.sleep(0.3)
    data = flir_image.capture(cam, tempsexp, DXff, DYff, Xinff, Yinff)
    C.append(Fonctions.corr2(data, reference_data))
    
plt.figure()
plt.plot(vec, C, 'r+') # 'r+-' : rouge, points en croix, reliés
plt.xlabel('Valeur vec[k]')
plt.ylabel('Corrélation')
plt.title('Corrélation')
plt.grid(True)
plt.show()

----------------------------------------------------------------------------------------------------------

Find $x_{\pi}$ and $x_{2\pi}$ :

In [14]:
value_pi, value_2pi = min(C), max(C)
xpi, x2pi = vec[C.index(value_pi)], vec[C.index(value_2pi)]
print(f"xpi = {xpi} and x2pi = {x2pi}")

xpi = 130 and x2pi = 235


Cosine fit:

# 2. 